# Phase 3: historical scorecard for the multi-cluster tracker

There is one tracker, not a single-cell tracker plus a multi-cell tracker. Every detected storm cluster receives a track; one cluster is simply the one-track case. This notebook tests whether its forecast for the Booz Allen McLean office agrees with a later, real KLWX radar scan.

Its output is a row per forecast origin and horizon: `P(any tracked 35 dBZ cluster covers McLean)` versus whether later radar actually showed 35 dBZ within 2 km of the office. This is a reflectivity-coverage baseline, not a severe-weather or damage probability.

In [ ]:
%pip install -e "../.[nexrad,dev]"

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import gc, json, re, sys

import boto3
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config

repo = Path.cwd().parent
sys.path.insert(0, str(repo / 'src'))
from storm_motion.evaluation import brier_score
from storm_motion.nexrad import load_lowest_reflectivity_sweep
from storm_motion.phase3 import evaluate_paths

catalog = json.loads((repo / 'config' / 'phase3_seed_catalog.json').read_text())
target = catalog['target']
bucket = 'unidata-nexrad-level2'
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))
print(f"Target: {target['name']} · observed label = {target['threshold_dbz']} dBZ within {target['label_radius_m'] / 1000:.0f} km")

In [ ]:
def select_regular_scans(event, station='KLWX', cadence_minutes=5):
    """Choose the closest distinct public Level II file at each cadence target."""
    start = datetime.fromisoformat(event['start_utc'].replace('Z', '+00:00'))
    end = datetime.fromisoformat(event['end_utc'].replace('Z', '+00:00'))
    prefix = f"{start:%Y/%m/%d}/{station}/"
    pattern = re.compile(rf'{station}(\d{{8}})_(\d{{6}})_V06$')
    available = []
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get('Contents', []):
            match = pattern.search(item['Key'])
            if match:
                observed_at = datetime.strptime(''.join(match.groups()), '%Y%m%d%H%M%S').replace(tzinfo=timezone.utc)
                if start <= observed_at <= end:
                    available.append((observed_at, item['Key']))
    if not available:
        raise RuntimeError(f"No {station} scans found for {event['event_id']}")
    targets, current = [], start
    while current <= end:
        targets.append(current)
        current += timedelta(minutes=cadence_minutes)
    selected, used = [], set()
    for desired in targets:
        observed_at, key = min(available, key=lambda item: abs(item[0] - desired))
        if key not in used:
            selected.append((observed_at, key))
            used.add(key)
    return selected

def download_event(event):
    data_dir = repo / 'data' / 'phase3' / event['event_id']
    data_dir.mkdir(parents=True, exist_ok=True)
    selected = select_regular_scans(event)
    paths = []
    for observed_at, key in selected:
        path = data_dir / Path(key).name
        if not path.exists():
            s3.download_file(bucket, key, str(path))
        paths.append(path)
    print(f"{event['event_id']}: {len(paths)} KLWX scans from {selected[0][0]:%H:%M} to {selected[-1][0]:%H:%M} UTC")
    return paths

In [ ]:
# Actual Phase 3 smoke run: one event at a time stays below the 4 GB RAM limit.
rows = []
for event in catalog['events']:
    event_paths = download_event(event)
    event_rows = evaluate_paths(
        event_id=event['event_id'], paths=event_paths, load_frame=load_lowest_reflectivity_sweep,
        target_latitude_deg=target['latitude_deg'], target_longitude_deg=target['longitude_deg'],
        horizons_minutes=catalog['horizons_minutes'], label_radius_m=target['label_radius_m'],
        threshold_dbz=target['threshold_dbz'],
    )
    rows.extend(row.as_dict() for row in event_rows)
    del event_paths, event_rows
    gc.collect()

if not rows:
    raise RuntimeError('No forecast/observation pairs were produced. Check the scan window and horizons.')
results = pd.DataFrame(rows).sort_values(['event_id', 'scan_time', 'horizon_minutes'])
results_path = repo / 'data' / 'phase3_smoke_results.jsonl'
results_path.parent.mkdir(parents=True, exist_ok=True)
results.to_json(results_path, orient='records', lines=True, date_format='iso')
print(f"Wrote {len(results)} forecast/observation rows to {results_path}")
results[['event_id', 'scan_time', 'horizon_minutes', 'active_track_count', 'combined_probability', 'observed_35dbz_coverage']].head(12)

In [ ]:
def score_group(group):
    probability = group['combined_probability'].tolist()
    observed = group['observed_35dbz_coverage'].tolist()
    predicted_yes = group['combined_probability'] >= 0.50
    actual_yes = group['observed_35dbz_coverage'].astype(bool)
    return pd.Series({
        'forecast_rows': len(group),
        'observations_with_35dbz': int(actual_yes.sum()),
        'brier_score_lower_is_better': brier_score(probability, observed),
        'hits_at_50pct': int((predicted_yes & actual_yes).sum()),
        'misses_at_50pct': int((~predicted_yes & actual_yes).sum()),
        'false_alarms_at_50pct': int((predicted_yes & ~actual_yes).sum()),
    })

scorecard = results.groupby(['event_id', 'horizon_minutes']).apply(score_group, include_groups=False)
display(scorecard)

if results['observed_35dbz_coverage'].nunique() < 2:
    print('Important: these seeds have only one observed outcome class. The pipeline works, but this is not enough evidence to judge calibration; Phase 3 needs historical McLean hits and non-hits before a model decision.')

## How to read it

- **Combined probability**: baseline chance that at least one tracked reflectivity footprint covers the office, using an explicit independence approximation.
- **Observed 35 dBZ coverage**: later KLWX radar label; it does not claim severe weather or damage at the office.
- **Brier score**: lower is better. A scorecard with only negatives can look deceptively good, which is why the next Phase 3 step expands to many historical hit and non-hit events.

Once this smoke run is sound, the exact evaluator becomes the EMR job: a larger event catalog in, partitioned evaluation rows and calibration reports in S3.